# slop-check: fine-tune on Kaggle (free T4) with Unsloth

**One-time setup**
1. Kaggle account must be **phone-verified** (needed for GPU + Internet).
2. On your Mac, zip the project *with* your dataset:
   `zip -r slop-check.zip slop-check -x 'slop-check/.venv/*' 'slop-check/runs/*' '*/__pycache__/*'`
3. kaggle.com → **Datasets → New Dataset** → upload `slop-check.zip` (keep it **Private**).
4. New Notebook → **File → Import Notebook** → this file. Then in the right panel:
   - **Add Input** → your dataset
   - Settings → Accelerator → **GPU T4 x1** (not P100: Unsloth needs a newer GPU)
   - Settings → Internet → **On**
   - Add-ons → Secrets → add `HF_TOKEN` (Hugging Face *write* token), toggle it on for this notebook
5. Edit the config cell, then **Run All**. Expect ~30–60 min training + ~10 min GGUF export.

In [ ]:
# ---- config ------------------------------------------------------------------
HF_REPO   = "your-hf-username/slop-check-1.5b-gguf"   # <- change me ("" = don't push)
BASE      = "unsloth/Qwen2.5-Coder-1.5B-Instruct"
MAX_STEPS = 30        # 30 = ~5 min smoke test. Set to -1 for the full run (2 epochs).
EPOCHS    = 2
GIT_REPO  = ""        # optional: clone code from GitHub instead of using the uploaded zip

In [ ]:
import os, shutil, glob, subprocess
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Unsloth trains on one GPU
WORK = "/kaggle/working/slop-check"

if GIT_REPO:
    subprocess.run(["git", "clone", "-q", GIT_REPO, WORK], check=True)
else:
    # find the uploaded project (folder that contains pyproject.toml) under /kaggle/input
    hits = [p for p in glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)]
    assert hits, "Upload slop-check.zip as a Kaggle Dataset and add it as an input"
    src = os.path.dirname(hits[0])
    shutil.copytree(src, WORK, dirs_exist_ok=True, ignore=shutil.ignore_patterns(".venv", "runs"))
os.chdir(WORK)
print("code:", WORK)
for f in ("data/out/train.jsonl", "data/out/valid.jsonl"):
    n = sum(1 for _ in open(f)) if os.path.exists(f) else 0
    print(f, n, "examples")
assert os.path.exists("data/out/train.jsonl"), "data/out/ missing — build it on your Mac and re-upload"
print(open("data/out/stats.json").read() if os.path.exists("data/out/stats.json") else "")

In [ ]:
!pip install -q unsloth
!pip install -q -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception as e:
    print("No HF_TOKEN secret — model will be saved locally only.", e)
    HF_REPO = ""

In [ ]:
push = f"--push {HF_REPO}" if HF_REPO and os.environ.get("HF_TOKEN") else ""
!python training/train_unsloth.py --data data/out --out runs/unsloth --base {BASE} \
    --epochs {EPOCHS} --max-steps {MAX_STEPS} --gguf q4_k_m {push}

**Check it works** on the sample PR and the validation set (CPU llama.cpp, same as the GitHub Action will use).

In [ ]:
!pip install -q "llama-cpp-python>=0.3.0" --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu
GGUF = sorted(glob.glob("runs/unsloth/**/*.gguf", recursive=True))[0]
print(GGUF)
!slopcheck review --diff data/seed/sample_pr.diff --no-rules --backend llamacpp --model {GGUF}

In [ ]:
print("=== rules baseline ===")
!python eval/evaluate.py --data data/out/valid.jsonl --limit 100
print("=== fine-tuned model alone ===")
!python eval/evaluate.py --data data/out/valid.jsonl --limit 100 --no-rules --backend llamacpp --model {GGUF}

**Download** (if you didn't push to Hugging Face): the zip appears under *Output* on the right.

In [ ]:
!cd runs/unsloth && zip -qr /kaggle/working/slop-check-model.zip lora $(dirname {GGUF} | sed 's|runs/unsloth/||') && ls -lh /kaggle/working/*.zip